In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
new_clusters = pd.read_csv(f'{indir}/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', header=0, index_col=0)
new_clusters.index = [f'{x}_indexed' for x in new_clusters.index]
new_clusters.shape

In [ ]:
chrom100k_adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/merged.processed.h5ad')

In [ ]:
chrom100k_adata.obs.index = [f'{x}_indexed' for x in chrom100k_adata.obs.index]
chrom100k_adata.obs['cluster'] = new_clusters.reindex(chrom100k_adata.obs.index)['pred_coarse']
mask = ~chrom100k_adata.obs['cluster'].isna()
chrom100k_adata = chrom100k_adata[mask]

In [ ]:
chrom100k_adata

In [ ]:
hexes = ['#1D8348', '#5B2C6F', '#922B21', '#F1C40F']
chrom100k_adata.uns['donor_colors'] = hexes

In [ ]:
final_chrom100k = chrom100k_adata[~chrom100k_adata.obs['cluster'].isna()]
sc.pl.umap(final_chrom100k, color=['donor', 'time_num', 'cluster'], ncols=2)

### load all of the feature sets

In [ ]:
train_size = 0.8
test_size = 0.2
model = OneVsRestClassifier(XGBClassifier(eval_metric='logloss', random_state=0))

In [ ]:
genes_adata = sc.read_h5ad(f'{projdir}/h5ad/pred_rna/genes_adata.h5ad')

In [ ]:
enh_adata = sc.read_h5ad(f'{projdir}/h5ad/pred_rna/enh_adata.h5ad')

In [ ]:
enh_adata.obs['donor'] = [x.split('-')[1][:3] for x in enh_adata.obs.index]
enh_adata.obs['time'] = [x.split('-')[1][3:] for x in enh_adata.obs.index]
enh_adata.obs['time_num'] = [int(x.split('D')[1]) for x in enh_adata.obs['time']]

In [ ]:
peaks_adata = sc.read_h5ad(f'{projdir}/h5ad/pred_rna/peaks_adata.h5ad')

In [ ]:
chrom5k_adata = sc.read_h5ad(f'{projdir}/h5ad/pred_rna/chrom5k_adata.h5ad')

In [ ]:
dmrs_adata = sc.read_h5ad(f'{projdir}/h5ad/pred_rna/dmrs_adata.h5ad')

In [ ]:
genes_adata.obs['cluster'] = new_clusters.reindex(genes_adata.obs.index)['pred_coarse']
mask = ~genes_adata.obs['cluster'].isna()
genes_adata = genes_adata[mask]

In [ ]:
enh_adata.obs['cluster'] = new_clusters.reindex(enh_adata.obs.index)['pred_coarse']
mask = ~enh_adata.obs['cluster'].isna()
enh_adata = enh_adata[mask]

In [ ]:
peaks_adata.obs['cluster'] = new_clusters.reindex(peaks_adata.obs.index)['pred_coarse']
mask = ~peaks_adata.obs['cluster'].isna()
peaks_adata = peaks_adata[mask]

In [ ]:
chrom5k_adata.obs['cluster'] = new_clusters.reindex(chrom5k_adata.obs.index)['pred_coarse']
mask = ~chrom5k_adata.obs['cluster'].isna()
chrom5k_adata = chrom5k_adata[mask]

In [ ]:
dmrs_adata.obs['cluster'] = new_clusters.reindex(dmrs_adata.obs.index)['pred_coarse']
mask = ~dmrs_adata.obs['cluster'].isna()
dmrs_adata = dmrs_adata[mask]

In [ ]:
chrom100k_adata.obs['cluster'].isna().sum(), \
chrom5k_adata.obs['cluster'].isna().sum(), \
enh_adata.obs['cluster'].isna().sum(), \
peaks_adata.obs['cluster'].isna().sum(), \
dmrs_adata.obs['cluster'].isna().sum()

In [ ]:
chrom100k_adata.shape, \
chrom5k_adata.shape, \
enh_adata.shape, \
peaks_adata.shape, \
dmrs_adata.shape

In [ ]:
# X = np.hstack([chrom100k_adata.obsm['X_pca'], chrom100k_adata.obs['time_num'].values.reshape(-1, 1)])
# y = chrom100k_adata.obs['cluster']
#
# X_train, X_test, y_train, y_test_chrom100k = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model.fit(X_train, y_train)
#
# chrom100k_pred = model.predict(X_test)
# print(classification_report(y_test_chrom100k, chrom100k_pred))

In [ ]:
# X = np.hstack([genes_adata.obsm['X_pca'], genes_adata.obs['time_num'].values.reshape(-1, 1)])
# y = genes_adata.obs['cluster']
#
# X_train, X_test, y_train, y_test_genes = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model.fit(X_train, y_train)
#
# genes_pred = model.predict(X_test)
# print(classification_report(y_test_genes, genes_pred))

In [ ]:
# X = np.hstack([enh_adata.obsm['X_pca'], enh_adata.obs['time_num'].values.reshape(-1, 1)])
# y = enh_adata.obs['cluster']
#
# X_train, X_test, y_train, y_test_enh = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model.fit(X_train, y_train)
#
# enh_pred = model.predict(X_test)
# print(classification_report(y_test_enh, enh_pred))

In [ ]:
# X = np.hstack([peaks_adata.obsm['X_pca'], peaks_adata.obs['time_num'].values.reshape(-1, 1)])
# y = peaks_adata.obs['cluster']
#
# X_train, X_test, y_train, y_test_peaks = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model.fit(X_train, y_train)
#
# peaks_pred = model.predict(X_test)
# print(classification_report(y_test_peaks, peaks_pred))

In [ ]:
# X = np.hstack([chrom5k_adata.obsm['X_pca'], chrom5k_adata.obs['time_num'].values.reshape(-1, 1)])
# y = chrom5k_adata.obs['cluster']
#
# X_train, X_test, y_train, y_test_chrom5k = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model.fit(X_train, y_train)
#
# chrom5k_pred = model.predict(X_test)
# print(classification_report(y_test_chrom5k, chrom5k_pred))

In [ ]:
# X = np.hstack([dmrs_adata.obsm['X_pca'], dmrs_adata.obs['time_num'].values.reshape(-1, 1)])
# y = dmrs_adata.obs['cluster']
#
# X_train, X_test, y_train, y_test_dmrs = train_test_split(X, y, test_size=test_size, train_size=train_size)
# model.fit(X_train, y_train)
#
# dmrs_pred = model.predict(X_test)
# print(classification_report(y_test_dmrs, dmrs_pred))

### one saved stratified split on nuclei shared by all feature sets (dmrs dropped)

In [ ]:
feature_sets = {'chrom100k': chrom100k_adata, 'chrom5k': chrom5k_adata, 'genes': genes_adata, 'peaks': peaks_adata, 'enh': enh_adata}
common = new_clusters.index
for k, a in feature_sets.items():
    common = common.intersection(a.obs.index)
common = common.sort_values()
len(common)

In [ ]:
y = new_clusters.loc[common, 'pred_coarse'].values
time_num = new_clusters.loc[common, 'time_num'].values.reshape(-1, 1)
train_idx, test_idx = train_test_split(np.arange(len(common)), test_size=test_size, train_size=train_size, stratify=y, random_state=0)
split = pd.DataFrame({'split': 'train'}, index=common)
split.iloc[test_idx, 0] = 'test'
split['split'].value_counts()

In [ ]:
preds = {}
for k, a in feature_sets.items():
    X = np.hstack([a.obsm['X_pca'][a.obs.index.get_indexer(common)], time_num])
    model.fit(X[train_idx], y[train_idx])
    preds[k] = model.predict(X[test_idx])
    print(k, f1_score(y[test_idx], preds[k], labels=tmp_order, average='macro'))

### tie it all together

In [ ]:
final_scores = pd.DataFrame({k: f1_score(y[test_idx], v, labels=tmp_order, average=None) for k, v in preds.items()}, index=tmp_order)
final_scores

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
split.to_csv(f'{outdir}/figure1/mc_cluster_f1_split.csv', sep='\t')
to_plot = final_scores.reset_index().melt(id_vars='index')
to_plot.columns = ['cluster', 'variable', 'value']
to_plot.to_csv(f'{outdir}/figure1/mc_cluster_f1_scores_seed0_melt.csv', sep='\t')

### what about raw features?

In [ ]:
# X_train, X_test, y_train, y_test_chrom100k = train_test_split(X, y, test_size=test_size, train_size=train_size)

In [ ]:
!date